In [1]:
%load_ext autoreload
%autoreload 2
import os
import matplotlib.pyplot as plt
import seaborn as sns
from os.path import join
from tqdm import tqdm
import pandas as pd
import sys
import joblib
from scipy.special import softmax
from neuro import config
import numpy as np
from collections import defaultdict
from copy import deepcopy
import pandas as pd
import neuro.sasc.viz
import neuro.viz
from neuro.sasc import analyze_helper
import neuro.sasc
from neuro.sasc.modules.fmri_module import convert_module_num_to_voxel_num
from scipy.stats import false_discovery_control
import dvu
dvu.set_style()

# pcs = joblib.load(join(FMRI_DIR, "voxel_neighbors_and_pcs", "loo_pc_UTS02.pkl"))
# pcs['good_voxels'].shape
# pcs['pca_projections'].shape

<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
/home/chansingh/automated-brain-explanations/.venv/lib/python3.12/site-packages/spacy/cli/_util.py:23: DeprecationWarning: Importing 'parser.split_arg_string' is deprecated, it will only be available in 'shell_completion' in Click 9.0.
  from click.parser import split_arg_string
/home/chansingh/automated-brain-explanations/.venv/lib/python3.12/site-packages/weasel/util/config.py:8: DeprecationWarning: Importing 'parser.split_arg_string' is deprecated, it will only be available in 'shell_completion' in Click 9.0.
  from click.parser import split_arg_string


In [3]:
pilot_name = 'pilot_story_data.pkl'
# pilot_name = 'pilot3_story_data.pkl'
# pilot_name = 'pilot4_story_data.pkl'
# pilot_name = "pilot6_story_data.pkl"
# pilot_name = "pilot8_story_data.pkl"

stories_data_dict = joblib.load(
    join(config.RESULTS_DIR_LOCAL, 'gct_processed', pilot_name))
if pilot_name == 'pilot_story_data.pkl':
    pilot_data_dir = join(config.PILOT_STORY_DATA_DIR, '20230504')
# elif pilot_name == 'pilot3_story_data.pkl':
#     pilot_data_dir = join(config.PILOT_STORY_DATA_DIR, '20231106')
# elif pilot_name == 'pilot4_story_data.pkl':
#     pilot_data_dir = join(config.PILOT_STORY_DATA_DIR, '20240509')
# elif pilot_name == 'pilot6_story_data.pkl':
#     pilot_data_dir = join(config.PILOT_STORY_DATA_DIR, '20241202')
# elif pilot_name == 'pilot7_story_data.pkl':
#     pilot_data_dir = join(config.PILOT_STORY_DATA_DIR, '20241204')
# elif pilot_name == 'pilot8_story_data.pkl':
#     pilot_data_dir = join(config.PILOT_STORY_DATA_DIR, '20241204')

In [6]:
# load responses
default_story_idxs = np.array([0, 1, 2, 3, 5])
# np.where(
    # (np.array(stories_data_dict['story_setting']) == 'default') |
    # (np.array(stories_data_dict['story_setting']) == 'roi')
# )[0]
resp_np_files = [stories_data_dict['story_name_new'][i].replace('_resps', '')
                 for i in default_story_idxs]
resps_dict = {
    k: np.load(join(pilot_data_dir, k))
    for k in tqdm(resp_np_files)
}

  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:00<00:00, 15.90it/s]


In [12]:
mats = defaultdict(list)
resp_chunks = defaultdict(list)
if pilot_name in ['pilot1']:
    use_clusters_list = [False, True]
else:
    use_clusters_list = [False]
for use_clusters in use_clusters_list:
    for story_num in default_story_idxs:
        rows = stories_data_dict["rows"][story_num]

        # get resp_chunks
        resp_story = resps_dict[
            stories_data_dict["story_name_new"][story_num].replace(
                '_resps', '')
        ].T  # (voxels, time)
        timing = stories_data_dict["timing"][story_num]
        if 'paragraphs' in stories_data_dict.keys():
            paragraphs = stories_data_dict["paragraphs"][story_num]
        else:
            paragraphs = stories_data_dict["story_text"][story_num].split(
                "\n\n")
        # paragraphs = stories_data_dict["story_text"][story_num].split("\n\n")
        if pilot_name in ['pilot3_story_data.pkl']:
            paragraphs = [neuro.sasc.analyze_helper.remove_repeated_words(
                p) for p in paragraphs]
        assert len(paragraphs) == len(
            rows), f"{len(paragraphs)} != {len(rows)}"
        resp_chunks = analyze_helper.get_resps_for_paragraphs(
            timing, paragraphs, resp_story, offset=2, validate=True,
            split_hyphens=pilot_name in ["pilot6_story_data.pkl", "pilot7_story_data.pkl", "pilot8_story_data.pkl"])
        assert len(resp_chunks) <= len(paragraphs)

        # calculate mat
        mat = np.zeros((len(rows), len(paragraphs)))
        for i in range(len(resp_chunks)):
            if use_clusters == False:
                # driving single voxel
                if 'voxel_num' in rows.columns:
                    mat[:, i] = resp_chunks[i][rows["voxel_num"].values].mean(
                        axis=1).flatten()
                elif 'voxel_nums' in rows.columns:
                    mat[:, i] = [resp_chunks[i][x].mean()
                                 for x in rows['voxel_nums']]
                    # resp_chunks[i][rows["voxel_nums"].values].mean(
                    # axis=1).flatten()

            elif use_clusters == True:
                for r in range(len(rows)):
                    cluster_nums = rows.iloc[r]["cluster_nums"]
                    if isinstance(cluster_nums, np.ndarray):
                        vals = resp_chunks[i][cluster_nums].flatten()
                        mat[r, i] = np.nanmean(vals)
                    else:
                        # print(cluster_nums)
                        mat[r, i] = np.nan
        mat[:, 0] = np.nan  # ignore the first column
        # print('mat', mat)

        # sort by voxel_num
        if 'voxel_num' in rows.columns:
            args = np.argsort(rows["voxel_num"].values)
        elif pilot_name in ["pilot6_story_data.pkl", "pilot7_story_data.pkl", "pilot8_story_data.pkl"]:
            args = np.argsort(rows["expl"].values)
        else:
            args = np.argsort(rows["roi"].values)
        mat = mat[args, :][:, args]
        mats[use_clusters].append(deepcopy(mat))

        # plt.imshow(mat)
        # plt.colorbar(label="Mean response")
        # plt.xlabel("Corresponding paragraph\n(Ideally, diagonal should be brighter)")
        # plt.ylabel("Voxel")
        # plt.title(f"{story_data['story_name_new'][story_num][3:-10]}")
        # plt.show()

if 'voxel_num' in rows.columns:
    rows = rows.sort_values(by="voxel_num")
elif pilot_name in ["pilot6_story_data.pkl", "pilot7_story_data.pkl", "pilot8_story_data.pkl"]:
    rows = rows.sort_values(by="expl")
else:
    rows = rows.sort_values(by="roi")
expls = rows["expl"].values


m = {}
for use_clusters in [False, True]:
    mats[use_clusters] = np.array(mats[use_clusters])  # (6, 17, 17)
    m[use_clusters] = np.nanmean(mats[use_clusters], axis=0)
    m['std'] = np.nanstd(mats[use_clusters], axis=0)

/tmp/ipykernel_2452566/4069830421.py:88: RuntimeWarning: Mean of empty slice
  m[use_clusters] = np.nanmean(mats[use_clusters], axis=0)
/home/chansingh/automated-brain-explanations/.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [17]:
# make a column of tuples of (expl, voxel_num)
col = [(expl, vn) for expl, vn in zip(rows['expl'], rows['voxel_num'])]

In [23]:
means

"Voxel to be driven (expl, voxel_num)","(emotion, 3947)","(surprise, 5389)","(rejection, 22840)","(laughter, 22994)","(food preparation, 23367)","(time, 23386)","(birthdays, 26228)","(negativity, 29226)","(emotional expression, 29688)","(death, 32670)","(moments, 35484)","(physical injury or trauma, 43198)","(measurements, 47687)","(communication, 50190)","(locations, 50254)","(hair and clothing, 59916)","(locations, 61584)"
"Driving paragraph (expl, voxel_num)",,,,,,,,,,,,,,,,,
"(emotion, 3947)",-0.112892,0.322698,0.373482,0.256646,-0.363802,0.257474,-0.326311,0.021376,0.261055,0.064296,-0.050662,0.207923,-0.160642,0.205893,-0.210852,-0.451341,-0.467118
"(surprise, 5389)",-0.275120,-0.099193,0.185577,0.338719,-0.308539,0.403755,-0.126021,0.063132,0.250110,0.100388,0.100162,0.098201,-0.223649,0.163261,-0.300047,-0.461932,-0.317263
"(rejection, 22840)",-0.037934,0.020640,0.126043,0.205105,-0.113945,0.231095,-0.168672,0.080806,-0.015965,-0.123044,-0.236343,0.008987,-0.029591,0.019134,-0.117936,-0.405143,-0.125378
"(laughter, 22994)",-0.224739,-0.271258,0.156496,0.173629,-0.049631,0.177156,-0.107050,0.075432,0.138342,0.000859,0.193455,0.403402,-0.103052,0.422985,-0.071935,-0.156917,-0.320041
"(food preparation, 23367)",-0.190533,-0.021006,-0.107672,-0.091000,0.751981,0.005155,0.024738,-0.136077,-0.145397,-0.041515,-0.204285,0.310905,0.294608,-0.200695,-0.056131,0.006498,-0.176645
"(time, 23386)",-0.420625,-0.295675,-0.282081,0.210460,0.190498,0.140531,0.125903,-0.320041,0.182222,-0.025794,-0.107259,-0.005921,0.062305,-0.081271,-0.029647,-0.346817,0.253349
"(birthdays, 26228)",-0.053158,-0.135752,-0.281636,-0.300735,0.174020,-0.052390,0.243115,-0.378872,0.059086,-0.089170,0.119203,0.224734,0.115309,0.065731,-0.065388,0.042041,0.215939
"(negativity, 29226)",0.012311,0.152785,0.291774,-0.004668,-0.143109,0.292326,-0.425164,0.160702,-0.065129,0.110155,-0.143958,0.194715,-0.040770,0.036391,-0.187122,-0.225132,-0.480285
"(emotional expression, 29688)",-0.187665,-0.288911,0.002059,0.144770,-0.352885,0.371921,-0.202908,-0.035046,0.512249,0.109845,0.419508,0.210968,0.095121,0.484299,-0.104303,-0.485702,-0.310557


In [22]:
means = pd.DataFrame(m[False], columns=col, index=col)
means.index.name = 'Driving paragraph (expl, voxel_num)'
means.columns.name = 'Voxel to be driven (expl, voxel_num)'
stds = pd.DataFrame(m['std'], columns=col, index=col)
stds.index.name = 'Driving paragraph (expl, voxel_num)'
stds.columns.name = 'Voxel to be driven (expl, voxel_num)'
joblib.dump({'means': means, 'stds': stds}, 'means_and_stds_5stories_audio_comparison.joblib')

['means_and_stds_5stories_audio_comparison.joblib']